(querying-collections)=

# Querying Collections with Lambdas and LINQ

This optional applied reading follows Lists and Dictionaries. You need loops, method calls, returned values, and `List<T>`; you do not need custom classes or a database. The standard Chapter 8 preview, lab, and homework do not require this reading.

Language-Integrated Query (LINQ) provides library operations for asking questions about collections. A dispatch desk might keep all recorded delivery amounts but report only the large ones in ascending order. Start by stating that result: retain amounts at least 30, order them, and leave the source list unchanged. A loop and a query can implement the same rule. LINQ is useful when the sequence of operations communicates the rule clearly; a loop is useful when intermediate state or control flow needs attention.[^linq-introduction]

```{contents}
:local:
:depth: 2
```


```{index} lambda expression
```

## Read a Lambda



A **lambda expression** writes a short operation where a method expects one. Its parameter appears before `=>`, and the returned expression appears after it:

```csharp
amount => amount >= 30m
```

Read this as “given an amount, return whether it is at least 30.” The name `amount` is local to that expression. It receives each candidate value; it does not refer to an undeclared variable from an earlier cell. C# infers its type from the collection and library method. The `m` suffix makes 30 a decimal literal.

Compare the familiar named method with the lambda below. Both answer the same yes/no question. `Where` calls the supplied test for each amount and keeps values for which it returns `true`; the final `ToList` collects those values in a new list. `using System.Linq;` makes these query methods available.


In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var amounts = new List<decimal> { 45m, 10m, 30m, 20m };
static bool IsLarge(decimal amount) { return amount >= 30m; }

var byMethod = amounts.Where(IsLarge).ToList();
var byLambda = amounts.Where(amount => amount >= 30m).ToList();
Console.WriteLine(string.Join(", ", byMethod));
Console.WriteLine(string.Join(", ", byLambda));


45, 30
45, 30


Both lines are `45, 30`: filtering preserves the retained values' relative order. The method name is passed without parentheses because `Where` needs the operation to call later, not a Boolean result calculated now. Chapter 13 explains the delegate types behind such policy arguments. Here, recognize the input and returned expression before using the compact form.


```{index} LINQ
```

## Filter, Transform, and Order



Use method calls in a chain when each result is the input to the next operation:

```text
source.Where(test).OrderBy(key).Select(transform).ToList()
```

| Operation | Meaning in the report |
| --- | --- |
| `Where(amount => amount >= 30m)` | Retain qualifying amounts. |
| `OrderBy(amount => amount)` | Order by each amount's own value, ascending. |
| `Select(amount => amount + 5m)` | Produce a new amount with a delivery charge added. |
| `ToList()` | Execute the sequence and store its results in a new list. |

`Select` returns transformed values; it does not assign those values back into the source list. Compare a loop-based report with the query:


In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

var amounts = new List<decimal> { 45m, 10m, 30m, 20m };
var byLoop = new List<decimal>();
foreach (decimal amount in amounts)
{
    if (amount >= 30m) byLoop.Add(amount + 5m);
}
byLoop.Sort();

var byQuery = amounts.Where(amount => amount >= 30m)
                     .OrderBy(amount => amount)
                     .Select(amount => amount + 5m)
                     .ToList();
Console.WriteLine("Loop: " + string.Join(", ", byLoop));
Console.WriteLine("Query: " + string.Join(", ", byQuery));
Console.WriteLine("Source: " + string.Join(", ", amounts));
Console.WriteLine($"Count: {byQuery.Count}; total: {byQuery.Sum()}");


Loop: 35, 50
Query: 35, 50
Source: 45, 10, 30, 20
Count: 2; total: 85


The qualifying amounts are 45 and 30. Sorting gives 30 and 45; adding five gives 35 and 50. Both implementations produce that report and preserve the source. The loop adds the same constant before sorting, which preserves numeric order for these values. Do not move an arbitrary transformation across a sort without checking whether it changes the ordering.

**Predict:** change the cutoff to 50. Both reports become empty, their count is zero, and their sum is zero. If you use `Average` instead of `Sum`, define an empty-report policy first: the non-nullable decimal overload of `Average` cannot average an empty sequence.


```{index} deferred execution; materialization
```

## Choose When to Materialize



A query variable usually describes work, rather than storing its answer. **Deferred execution** means that operations such as `Where` read the source when the result is traversed. **Materialization** executes the query and stores the resulting values, for example with `ToList` or `ToArray`. A later traversal of the original query may therefore see changed source values.[^linq-introduction]

Predict the three output lines before running this standalone example:


In [3]:
using System;
using System.Collections.Generic;
using System.Linq;

var amounts = new List<decimal> { 45m, 10m, 30m };
var query = amounts.Where(amount => amount >= 30m);
var snapshot = query.ToList();
amounts[0] = 15m;
Console.WriteLine("Query now: " + string.Join(", ", query));
Console.WriteLine("Stored report: " + string.Join(", ", snapshot));
Console.WriteLine("Source now: " + string.Join(", ", amounts));


Query now: 30
Stored report: 45, 30
Source now: 15, 10, 30


The snapshot stores 45 and 30 before the edit. The later query traversal sees 15 at the first position and excludes it, so it returns only 30. Moving the edit before `ToList` would make the stored report contain only 30 too.

Choose a stored report when later source changes must not alter its recorded values. Here the values are decimals, so copying them is sufficient. A new list of mutable objects would still contain references to those objects; Chapter 11 develops that distinction. Keep lambda tests free of source mutations: changing a list while traversing it can invalidate its enumerator.

Not every query operation waits. `Sum`, `Count()`, and `Average` calculate a result immediately; `OrderBy` is deferred but must gather and order its input before producing the first result. Chapter 13 revisits enumeration and deferred work in more detail.


```{index} LINQ
```

## Practice: Prepare a Dispatch Report



A dispatch desk records amounts `25, -1, 40, 25, 10`. Negative values are invalid records for this report. Produce a new ascending list containing valid amounts, with a charge of five added to each. Keep duplicates and leave the source unchanged. Before running, predict the report count and sum. Then check empty input, all-invalid input, and a value of zero.


In [4]:
using System;
using System.Collections.Generic;
using System.Linq;

// exercise-id: cscs-collections-linq-dispatch
var amounts = new List<decimal> { 25m, -1m, 40m, 25m, 10m };
// Your code starts here
// Build a separate report, then print the report, count, and total.
// Your code ends here


In [5]:
using System;
using System.Collections.Generic;
using System.Linq;

var amounts = new List<decimal> { 25m, -1m, 40m, 25m, 10m };
var report = amounts.Where(amount => amount >= 0m)
                    .OrderBy(amount => amount)
                    .Select(amount => amount + 5m)
                    .ToList();
Console.WriteLine(string.Join(", ", report));
Console.WriteLine($"Count: {report.Count}; total: {report.Sum()}");
Console.WriteLine("Source: " + string.Join(", ", amounts));


15, 30, 30, 45
Count: 4; total: 120
Source: 25, -1, 40, 25, 10


The report is `15, 30, 30, 45`, with count four and sum 120. The duplicate 25 amounts remain two separate entries. Empty or all-invalid input gives an empty report and sum zero. A zero amount is valid and becomes five; changing the test to `> 0m` would incorrectly discard it.

Choose the loop version when it helps you inspect intermediate state or express a more complex rule. Choose the query version when filtering, ordering, and transformation make the report's contract easier to read. Verify either version against independently predicted results.


```{rubric} Footnotes
```

[^linq-introduction]: Microsoft Learn, [Introduction to LINQ Queries](https://learn.microsoft.com/en-us/dotnet/csharp/linq/get-started/introduction-to-linq-queries), consulted October 21, 2026. The examples and dispatch-report exercise here are original. The reference distinguishes query construction, execution, and stored results.
